In [8]:
# %%
# Prepare dataframe from monthly rasters for prediction over the entire grid
# we use the fitted model from A3

# V1 - 2026-09-21 - create code - jpk

# WORKFLOW
#  Load all predictor rasters for the month
#  Stack and create a df
#  Format df to fit column requirement of the model


In [9]:
# LIBRARIES
import os
import gc
import glob
import numpy as np
import pandas as pd 
import xarray as xr
import rioxarray as rxr
import pyarrow as pa    
import pyarrow.parquet as pq
from tqdm import tqdm
import logging

# CONFIGURATIONS
start_year, start_month, start_day = 2001, 1, 1
end_year,   end_month, end_day   = 2025, 12, 31

static_paths = {
    "elev_mean":       r"F:\ClimateDataWS\Data\Datasets\DEM_1km\DEM_1km_elevation_mean.tif",
    "elev_std":        r"F:\ClimateDataWS\Data\Datasets\DEM_1km\DEM_1km_elevation_std.tif",
    "elev_max":        r"F:\ClimateDataWS\Data\Datasets\DEM_1km\DEM_1km_elevation_max.tif",
    "elev_min":        r"F:\ClimateDataWS\Data\Datasets\DEM_1km\DEM_1km_elevation_min.tif",
    "slope_mean":      r"F:\ClimateDataWS\Data\Datasets\DEM_1km\DEM_1km_slope_mean.tif",
    "aspect_mean":     r"F:\ClimateDataWS\Data\Datasets\DEM_1km\DEM_1km_aspect_mean.tif",
    "elev_smoothed":   r"F:\ClimateDataWS\Data\Processed\Fetch\Preprocessing\dem_smoothed.tif",
    "fetch_1000m_dist":r"F:\ClimateDataWS\Data\Processed\Fetch\Preprocessing\fetch_dist_1000m.tif",
    "fetch_1000m_dir": r"F:\ClimateDataWS\Data\Processed\Fetch\Preprocessing\fetch_dir_1000m.tif",
    "fetch_500m_dist": r"F:\ClimateDataWS\Data\Processed\Fetch\Preprocessing\fetch_dist_500m.tif",
    "fetch_500m_dir":  r"F:\ClimateDataWS\Data\Processed\Fetch\Preprocessing\fetch_dir_500m.tif",
    "local_relief":    r"F:\ClimateDataWS\Data\Processed\Fetch\Preprocessing\local_relief.tif",
}

template = rxr.open_rasterio(r"F:\ClimateDataWS\Data\Datasets\DEM_1km\DEM_1km_elevation_max.tif")
out_base = r"F:\ClimateDataWS\Data\Processed\Monthly_downscaling"

monthly_raster_temp = r"F:\ClimateDataWS\Data\Processed\Monthly_downscaling\Input_rasters"
tmax_temp_dir = os.path.join(out_base, 'Temp_tmax')
tmax_dynamic_pred_dir = os.path.join(tmax_temp_dir, 'Dynamic_pred')
tmax_yearly_parquet_dir = os.path.join(tmax_temp_dir, 'Tmax_yearly_parquet')

tmin_temp_dir = os.path.join(out_base, 'Temp_tmin')
tmin_dynamic_pred_dir = os.path.join(tmin_temp_dir, 'Dynamic_pred')
tmin_yearly_parquet_dir = os.path.join(tmin_temp_dir, 'Tmin_yearly_parquet')

static_dir = os.path.join(out_base, "Static_df")

for d in [tmax_temp_dir, tmax_dynamic_pred_dir, tmax_yearly_parquet_dir, 
    static_dir, tmin_temp_dir, tmin_dynamic_pred_dir, tmin_yearly_parquet_dir]:
    if not os.path.exists(d):
        os.makedirs(d)

In [10]:
## HELPERS

def month_cyclical(month: int):
    """Return (sin, cos) encoding for a calendar month (1–12)."""
    angle = 2 * np.pi * month / 12
    return np.sin(angle), np.cos(angle)
 
def mlabel(year: int, month: int) -> str:
    return f"{year}{month:02d}"

# Monthly raster path dictionary; divide that are in temp folder vs in prec folder

def build_monthly_paths(year: int, month: int, t_variable: str) -> dict:
    """
    returns a dictionary of {feature_name: filepath} for all monthly rasters
    within the temperature folder (gridded temperature products)
    for the given year-month-variable (variable: tmax or tmin)
    """
    L = mlabel(year, month)
    R = monthly_raster_temp
    return {
        #Temperature average variables
        f"chirts_{t_variable}_average": f"{R}\\chirts\\chirts_{t_variable}_average_{L}.tif",
        f"era5l_{t_variable}_average": f"{R}\\era5l\\era5l_{t_variable}_average_{L}.tif",
        f"har_{t_variable}_average": f"{R}\\har\\har_{t_variable}_average_{L}.tif",

        #Temperature secondary variables from CHIRTS 
        f"chirts_{t_variable}_max": f"{R}\\chirts\\chirts_{t_variable}_max_{L}.tif",
        f"chirts_{t_variable}_min": f"{R}\\chirts\\chirts_{t_variable}_min_{L}.tif",
        f"chirts_{t_variable}_std": f"{R}\\chirts\\chirts_{t_variable}_std_{L}.tif",
        f"chirts_dtr_std": f"{R}\\chirts\\chirts_dtr_std_{L}.tif", 

        #GLDAS weather variables
        'gldas_air_temp':   f"{R}\\gldas_air_temp\\gldas_air_temp_monthly_mean_{L}.tif",
        'gldas_lwdown': f"{R}\\gldas_lwdown\\gldas_lwdown_monthly_sum_{L}.tif",
        'gldas_specific_humidity': f"{R}\\gldas_specific_humidity\\gldas_specific_humidity_monthly_mean_{L}.tif",
        'gldas_surface_pressure': f"{R}\\gldas_surface_pressure\\gldas_surface_pressure_monthly_mean_{L}.tif",
        'gldas_swdown': f"{R}\\gldas_swdown\\gldas_swdown_monthly_sum_{L}.tif",

        # MODIS LST
        "lst_d_monthly_mean":       f"{R}\\modis_lst\\lst_d_monthly_mean_{L}.tif",
        "lst_d_missing_fraction":   f"{R}\\modis_lst\\lst_d_missing_fraction_{L}.tif",
        "lst_n_monthly_mean":       f"{R}\\modis_lst\\lst_n_monthly_mean_{L}.tif",
        "lst_n_missing_fraction":   f"{R}\\modis_lst\\lst_n_missing_fraction_{L}.tif",
        "lst_diff_monthly_mean":    f"{R}\\modis_lst\\lst_diff_monthly_mean_{L}.tif",
        "lst_diff_missing_fraction":f"{R}\\modis_lst\\lst_diff_missing_fraction_{L}.tif"
}

SEASONS = {
    'winter':       [12, 1, 2],
    'pre_monsoon':  [3, 4, 5],
    'monsoon':      [6, 7, 8, 9],
    'post_monsoon': [10, 11],
}

def get_season(month):
    for s, months in SEASONS.items():
        if month in months:
            return s
    return None

SEASONS_NUM = {
    1: 'winter',
    2: 'pre_monsoon',
    3: 'monsoon',
    4: 'post_monsoon'
}

def get_season_num(season_str):
    for s_n, s_name in SEASONS_NUM.items():
        if s_name == season_str:
            return s_n
    return None

In [11]:
# 1 Build a static csv
static_csv = os.path.join(static_dir, "static_df.csv")
 
if os.path.exists(static_csv):
    print("Loading cached static df...")
    static_df = pd.read_csv(static_csv)
else:
    print("Building static df from rasters...")
    aligned = []
    for name, path in static_paths.items():
        da = rxr.open_rasterio(path, masked=True).rio.write_crs("EPSG:4326")
        da = da.rio.reproject_match(template).squeeze("band", drop=True)
        da.name = name
        aligned.append(da)
    static_xr = xr.merge(aligned, compat="override")
    static_df  = static_xr.to_dataframe().reset_index()
    
    # Save unrounded if you ever need raw, but we round below
    static_df.to_csv(static_csv, index=False)
    del aligned, static_xr
    gc.collect()
 
# Rounding spatial coordinates to ensure exact inner merge
static_df[["x", "y"]] = static_df[["x", "y"]].round(5)
print(f"Static df: {len(static_df):,} pixels")

Building static df from rasters...
Static df: 484,619 pixels


In [12]:
# 2 Build Monthly Loop and Create Frames

month_range = pd.period_range(
    start = f"{start_year}-{start_month:02d}",
    end = f"{end_year}-{end_month:02d}",
    freq = "M"
)

print(f"\n ============ Processing {len(month_range)} months for Tmax =========")

for t_variable in ['tmax', 'tmin']:
    logging.info(f"============ Processing {len(month_range)} days for {t_variable.upper()} ============")
    for period in tqdm(month_range, desc = f"Monthly feature extraction ({t_variable})"):
        year = period.year
        month = period.month
        label = mlabel(year, month)
        
        # Write directly to the dynamic pred directory to match Cell 5
        if t_variable == 'tmax':
            pred_out = os.path.join(tmax_dynamic_pred_dir, f"tmax_{label}.parquet")
        else:
            pred_out = os.path.join(tmin_dynamic_pred_dir, f"tmin_{label}.parquet")

        if os.path.exists(pred_out):
            continue

        paths = build_monthly_paths(year, month, t_variable)
        aligned = []
        missing_vars = []

        for name, path in paths.items():
            if not os.path.exists(path):
                missing_vars.append(name)
                continue
            da = rxr.open_rasterio(path, masked=True).rio.write_crs("EPSG:4326")
            da = da.rio.reproject_match(template).squeeze("band", drop = True)
            da.name = name
            aligned.append(da.load())
            da.close()

        if missing_vars:
            tqdm.write(f" [{label}] WARNING: Missing rasters: {missing_vars}")
        
        if not aligned:
            tqdm.write(f" [{label}] WARNING: No rasters found - skipping")
            continue

        dynamic_xr = xr.merge(aligned, compat='override')
        dynamic_df = dynamic_xr.to_dataframe().reset_index()
        
        # FIX: Round dynamic x/y identical to static x/y to prevent merge from failing 
        dynamic_df[["x", "y"]] = dynamic_df[["x", "y"]].round(5)

        # variable without rasters gets Nan Column
        for name in missing_vars:
            dynamic_df[name] = np.nan

        del aligned, dynamic_xr
        gc.collect()

        # merge with static df
        monthly_grid = dynamic_df.merge(static_df, on = ['x', 'y'], how = "inner")

        # Add temporal features - cyclical month columns
        sin_m, cos_m = month_cyclical(month)
        monthly_grid["month_sin"] = sin_m
        monthly_grid["month_cos"] = cos_m
        monthly_grid['year'] = year
        monthly_grid['month'] = month
        monthly_grid['season_num'] = (monthly_grid['month'].apply(get_season)).apply(get_season_num) 

        # Write directly to Parquet - saves time and space!
        monthly_grid.to_parquet(pred_out, index=False)

        del monthly_grid
        gc.collect()


 ============ Processing 300 months for Tmax =========


Monthly feature extraction (tmax):   0%|          | 0/300 [00:00<?, ?it/s]

Monthly feature extraction (tmin): 100%|██████████| 300/300 [20:40<00:00,  4.13s/it]


In [13]:
# 3 aggregate monthly parquets to yearly parquet

def aggregate_to_parquet(monthly_dir: str, parquet_dir: str, t_variable: str) -> str:
    """
    Read monthly parquets year by year, write to a single yearly parquet.
    Yields a list of yearly parquets.
    """
    parquet_paths = []

    for year in range(start_year, end_year + 1):
        yearly_path = os.path.join(parquet_dir, f"full_{year}.parquet")
        monthly_periods = pd.period_range(
            start = f"{year}-01-01", end = f"{year}-12-31", freq = "M"
        )
        expected = [
            os.path.join(monthly_dir, f"{t_variable}_{mlabel(p.year, p.month)}.parquet")
            for p in monthly_periods
        ]
        found = [f for f in expected if os.path.exists(f)]

        if not found:
            logging.warning(f"[{t_variable}] | {year} - no monthly files found, skipping")
            continue
        
        yearly_dfs = [pd.read_parquet(fpath) for fpath in found]
        combined_year = pd.concat(yearly_dfs, ignore_index=True)
        combined_year.to_parquet(yearly_path, index=False)
        parquet_paths.append(yearly_path)
        print(f"    [{t_variable}] {year} - {len(found)} months written to parquet")

    if not parquet_paths:
        logging.warning(f"[{t_variable}] No data in parquet paths - final df not written")
        return None
        
    logging.info(f"[{t_variable}] Yearly files aggregated successfully in {parquet_dir}.")
    logging.info(f"[{t_variable}] Load entire dataset memory-safely using: pd.read_parquet(r'{parquet_dir}')")
    
    return parquet_paths


logging.info("================ Aggregating to yearly parquet files ================")

tmax_pred_final_paths = aggregate_to_parquet(monthly_dir = tmax_dynamic_pred_dir, 
    parquet_dir = tmax_yearly_parquet_dir, 
    t_variable= 'tmax')

tmin_pred_final_paths = aggregate_to_parquet(monthly_dir = tmin_dynamic_pred_dir, 
    parquet_dir = tmin_yearly_parquet_dir, 
    t_variable= 'tmin')

logging.info("================ Process Complete! ================")

    [tmax] 2001 - 12 months written to parquet
    [tmax] 2002 - 12 months written to parquet
    [tmax] 2003 - 12 months written to parquet
    [tmax] 2004 - 12 months written to parquet
    [tmax] 2005 - 12 months written to parquet
    [tmax] 2006 - 12 months written to parquet
    [tmax] 2007 - 12 months written to parquet
    [tmax] 2008 - 12 months written to parquet
    [tmax] 2009 - 12 months written to parquet
    [tmax] 2010 - 12 months written to parquet
    [tmax] 2011 - 12 months written to parquet
    [tmax] 2012 - 12 months written to parquet
    [tmax] 2013 - 12 months written to parquet
    [tmax] 2014 - 12 months written to parquet
    [tmax] 2015 - 12 months written to parquet
    [tmax] 2016 - 12 months written to parquet
    [tmax] 2017 - 12 months written to parquet
    [tmax] 2018 - 12 months written to parquet
    [tmax] 2019 - 12 months written to parquet
    [tmax] 2020 - 12 months written to parquet
    [tmax] 2021 - 12 months written to parquet
    [tmax] 20

In [14]:
# ── 4. Clean up intermediate files ──────────────────────────────────────────────

import shutil

print("\n" + "=" * 60)
print("Cleaning up intermediate files and directories...")
print("=" * 60)

dirs_to_remove = [tmax_dynamic_pred_dir, tmin_dynamic_pred_dir, static_dir]

for d in dirs_to_remove:
    if os.path.exists(d):
        shutil.rmtree(d)
        print(f"Removed directory: {d}")

print("\nCleanup complete.")


Cleaning up intermediate files and directories...
Removed directory: F:\ClimateDataWS\Data\Processed\Monthly_downscaling\Temp_tmax\Dynamic_pred
Removed directory: F:\ClimateDataWS\Data\Processed\Monthly_downscaling\Temp_tmin\Dynamic_pred
Removed directory: F:\ClimateDataWS\Data\Processed\Monthly_downscaling\Static_df

Cleanup complete.
